# Monolayer Gap Analysis from Cell Labels
- *Isobel Taylor-Hearn, 2026*
- Measures every cell and every gap in a 2D endothelial monolayer, using
    - the micro-sam cell labels (one label tif per image, made on max projections). These are cleaned up first: fragments smaller than a cell (left where micro-sam's tiles meet) are merged into their neighbouring cell, and holes inside a cell are filled.
    - the original z-stacks, with channels in the same order for every experiment: C1 DAPI, C2 VE-Cad, C3 ICAM1 (see `reorder_channels_folder` in `General/convert_to_tifs.ipynb`)
- Per cell (cells touching the image border are excluded):
    - `cell_area_um2`
    - `cell_aspect_ratio`: longest / shortest axis of the ellipse fitted to the cell (1 = round, higher = more elongated)
    - `ICAM1_total`, `VECad_total`: summed over every z slice inside the cell
    - `VECad_junction`: how bright the VE-Cad junction around the cell is (see `measure_junction_vecad`)
- Per image: `n_cells`, `gap_area_pct` (% of the monolayer), `total_gap_area_um2`
- Intensities are raw values (no rescaling), but are only comparable **within** one experiment - each experiment used different fluorophores and microscope settings

In [ ]:
import pathlib
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from scipy import ndimage as ndi
from skimage import exposure, util
from skimage.color import label2rgb
from skimage.measure import label, regionprops, regionprops_table
from skimage.morphology import disk, remove_small_objects
from skimage.segmentation import clear_border, find_boundaries

import warnings
warnings.filterwarnings("ignore")

import contextlib
import joblib
from tqdm import tqdm
from joblib import Parallel, delayed

In [ ]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """
    Enables parallel jobs to run and display of a tqdm progress bar.

    Parameters:
    -----------
    tqdm_object : tqdm
        The tqdm progress bar instance to be updated.

    """
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

In [ ]:
def tif_pixel_sizes(tif):
    """Return (z_um, y_um, x_um) from a tifffile TiffFile; any value is None if unavailable. (From General/convert_to_tifs.ipynb)"""
    z_um = y_um = x_um = None
    tags = {t.name: t.value for t in tif.pages[0].tags.values()}
    xr = tags.get("XResolution")
    yr = tags.get("YResolution")
    if xr and xr[0]:
        x_um = xr[1] / xr[0]  # XResolution = pixels per unit -> um per pixel
    if yr and yr[0]:
        y_um = yr[1] / yr[0]
    ij = tif.imagej_metadata or {}
    if "spacing" in ij:
        z_um = float(ij["spacing"])
    return z_um, y_um, x_um

In [ ]:
# Start of the file name -> experiment. The rest of the file name describes the treatment.
EXPERIMENTS = {
    "04032025_HMVEC_VECAD488_ICAM1594_ZO1_647_": "04032025_HMVEC",
    "20240814_Akinola_HMVEC_ICAM488_VECADH647_ENOS594_": "20240814_Akinola",
    "Dorota_to_analzye_size_spaces_": "Dorota",
}


def add_image_details(df, filename):
    """
    Adds experimental details extracted from the filename to a dataframe.

    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to which image metadata will be added.
    filename : str
        The filename of the image, used to extract experimental details.

    Returns:
    --------
    df : pandas.DataFrame
        The updated dataframe with the following added columns:
        - 'filename': The original filename.
        - 'experiment': 04032025_HMVEC, 20240814_Akinola or Dorota
        - 'treatment': control, or the treatments joined with " + " (always in the order Fulv, E2P4, TNF / TNF24h)
    """
    df["filename"] = filename
    df["experiment"] = "UNKNOWN!"
    condition = filename
    for prefix, experiment in EXPERIMENTS.items():
        if filename.startswith(prefix):
            df["experiment"] = experiment
            condition = filename[len(prefix):]

    # The experiments spell treatments differently: e2_p4 / E2P4, tnfa / TNF
    condition = condition.lower().replace("e2_p4", "e2p4").replace("tnfa", "tnf")
    treatments = []
    if "fulv" in condition:
        treatments.append("Fulv")
    if "e2p4" in condition:
        treatments.append("E2P4")
    if "tnf24h" in condition:
        treatments.append("TNF24h")
    elif "tnf" in condition:
        treatments.append("TNF")
    df["treatment"] = " + ".join(treatments) if treatments else "control"

    return df

In [ ]:
def merge_small_fragments(cell_labels, pixel_area_um2, min_cell_area_um2=300):
    """
    Merge label fragments that are too small to be a cell into the neighbouring cell they share the longest border with.

    micro-sam segments big images in tiles, which sometimes cuts a cell into a big piece and a small fragment (often
    with a straight edge), or leaves a small detached bit of a cell. Removing the fragments instead would leave
    unlabelled areas that are counted as gaps. Fragments that don't touch any cell are removed.

    Parameters
    ----------
    cell_labels : ndarray (int)
        Cell labels, 0 = no cell.
    pixel_area_um2 : float
        Area of one pixel in µm².
    min_cell_area_um2 : float, optional (default=300)
        Labels smaller than this are fragments, not cells.

    Returns
    -------
    merged_labels : ndarray (int)
        Cell labels without fragments.
    """
    merged_labels = label(cell_labels)  # every separate piece gets its own label, so detached bits of a cell are fragments too
    fragments = [region for region in regionprops(merged_labels) if region.area * pixel_area_um2 < min_cell_area_um2]
    for fragment in sorted(fragments, key=lambda region: region.area):  # smallest first
        fragment_mask = merged_labels == fragment.label
        border = ndi.binary_dilation(fragment_mask) & ~fragment_mask  # pixels just outside the fragment
        neighbours = merged_labels[border]
        neighbours = neighbours[neighbours > 0]
        merged_labels[fragment_mask] = np.bincount(neighbours).argmax() if len(neighbours) else 0
    return merged_labels

In [ ]:
def fill_holes_in_cells(cell_labels):
    """
    Fill holes inside each cell, so they count as part of the cell rather than as gaps.

    A hole is an unlabelled area completely surrounded by one cell. Unlabelled areas between two or more cells are
    gaps and are left as they are.

    Parameters
    ----------
    cell_labels : ndarray (int)
        Cell labels, 0 = no cell.

    Returns
    -------
    filled_labels : ndarray (int)
        Cell labels with the holes inside each cell filled.
    """
    filled_labels = cell_labels.copy()
    for cell in regionprops(cell_labels):
        min_row, min_col, max_row, max_col = cell.bbox
        crop = filled_labels[min_row:max_row, min_col:max_col]  # a view: changing crop changes filled_labels
        crop[cell.image_filled & (crop == 0)] = cell.label
    return filled_labels

In [ ]:
def find_gaps(cell_labels, pixel_area_um2, min_gap_area_um2=4):
    """
    Find gaps in the monolayer: unlabelled areas that are completely enclosed by cells.

    Neighbouring cell labels touch each other, so any enclosed unlabelled area is a gap. Gaps smaller than
    `min_gap_area_um2` are ignored. Unlabelled areas touching the image border are not counted as gaps, as they
    can't be told apart from the edge of the monolayer.

    Parameters
    ----------
    cell_labels : ndarray (int)
        Cell labels, 0 = no cell.
    pixel_area_um2 : float
        Area of one pixel in µm².
    min_gap_area_um2 : float, optional (default=4)
        Smallest gap that is counted, in µm².

    Returns
    -------
    gap_labels : ndarray (int)
        Labelled gaps.
    monolayer : ndarray (bool)
        Cells + gaps.
    """
    cells = cell_labels > 0
    monolayer = ndi.binary_fill_holes(cells)
    gaps = monolayer & ~cells
    gaps = remove_small_objects(gaps, min_size=int(min_gap_area_um2 / pixel_area_um2))
    return label(gaps), monolayer

In [ ]:
def measure_junction_vecad(vecad, cell_labels, cell_ids, search_radius_px):
    """
    How bright the VE-Cad junction around each cell is.

    The label outlines don't sit exactly on the VE-Cad junctions (they are often 1-3 pixels off). So for every pixel on
    a cell's outline we take the brightest VE-Cad within `search_radius_px` - this finds the junction wherever it is
    close by - and then take the median of these values around the whole outline.

    Parameters
    ----------
    vecad : ndarray
        VE-Cad image (sum over z).
    cell_labels : ndarray (int)
        Cell labels, 0 = no cell.
    cell_ids : array of int
        The cells to measure.
    search_radius_px : int
        How far from the outline to look for the junction, in pixels.

    Returns
    -------
    ndarray
        VE-Cad junction brightness, one value per cell in `cell_ids`.
    """
    local_peak = ndi.maximum_filter(vecad, footprint=disk(search_radius_px))
    outlines = np.where(find_boundaries(cell_labels, mode="inner"), cell_labels, 0)  # outline pixels keep their cell's label
    return np.array(ndi.median(local_peak, labels=outlines, index=cell_ids))

In [ ]:
def rescale_for_display(image, low_percentile=1, high_percentile=99.8):
    """Stretch an image to 0-1 between two percentiles. For plots only - measurements always use the raw values."""
    p_low, p_high = np.percentile(image, [low_percentile, high_percentile])
    return exposure.rescale_intensity(image.astype(np.float32), in_range=(p_low, p_high), out_range=(0, 1))


def plot_qc(output_dir, filename, dapi, vecad, icam1, cell_labels, measured_labels, gap_labels, monolayer, per_cell, gap_area_pct):
    """
    Save a QC figure for one image to output_dir/qc:
        top row:    original (sum over z) | cell labels | gaps
        bottom row: zoom of VE-Cad with the label outlines | VE-Cad junction per cell | ICAM1 per µm² per cell
    """
    rgb = np.stack([rescale_for_display(vecad), rescale_for_display(icam1), rescale_for_display(dapi)], axis=-1)
    outlines = find_boundaries(cell_labels, mode="inner")

    fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(18, 12))

    ax[0, 0].imshow(rgb)
    ax[0, 0].set_title("original (sum over z)\nVE-Cad red, ICAM1 green, DAPI blue")

    ax[0, 1].imshow(label2rgb(measured_labels, image=rgb, bg_label=0, alpha=0.3))
    ax[0, 1].imshow(np.ma.masked_where(~outlines, outlines), cmap="gray", interpolation="none")
    ax[0, 1].set_title(f"cell labels: {len(per_cell)} cells measured\nuncoloured cells touch the image border - not measured")

    gap_overlay = rgb.copy()
    gap_overlay[~monolayer] = [0.3, 0.3, 0.3]
    gap_overlay[gap_labels > 0] = [1, 1, 0]
    ax[0, 2].imshow(gap_overlay)
    ax[0, 2].set_title(f"gaps (yellow): {gap_area_pct:.1f}% of monolayer, n = {gap_labels.max()}\ngrey = outside the monolayer, not counted")

    height, width = vecad.shape
    zoom = (slice(3 * height // 8, 5 * height // 8), slice(3 * width // 8, 5 * width // 8))  # centre of the image
    ax[1, 0].imshow(rescale_for_display(vecad)[zoom], cmap="gray")
    ax[1, 0].imshow(np.ma.masked_where(~outlines[zoom], outlines[zoom]), cmap="autumn", alpha=0.5, interpolation="none")
    ax[1, 0].set_title("zoom (centre): VE-Cad + label outlines")

    # Averages, not totals, so big cells don't stand out just for being big
    values_per_cell = {
        "VE-Cad junction (median along the outline)": per_cell["VECad_junction"],
        "ICAM1 per µm² (ICAM1_total / cell_area_um2)": per_cell["ICAM1_total"] / per_cell["cell_area_um2"],
    }
    cell_ids = per_cell["cell_label"].to_numpy().astype(measured_labels.dtype)
    for a, (title, values) in zip([ax[1, 1], ax[1, 2]], values_per_cell.items()):
        value_map = util.map_array(measured_labels, cell_ids, values.to_numpy())
        im = a.imshow(np.ma.masked_where(measured_labels == 0, value_map), cmap="viridis", interpolation="none")
        fig.colorbar(im, ax=a, fraction=0.046, pad=0.02)
        a.set_title(title)

    for a in ax.ravel():
        a.axis("off")
    fig.suptitle(f"{filename}\n{per_cell['experiment'].iloc[0]}: {per_cell['treatment'].iloc[0]}" if len(per_cell) else filename)
    fig.tight_layout()
    for attempt in range(10):  # Dropbox briefly locks files while it syncs them, so overwriting can fail - try again
        try:
            fig.savefig(output_dir / "qc" / f"{filename}_qc.png", dpi=150, bbox_inches="tight")
            break
        except OSError:
            time.sleep(3)
    plt.close(fig)

In [ ]:
def quantify_image(stack_path, labels_dir, output_dir, to_plot=True, dapi_channel=0, vecad_channel=1, icam1_channel=2,
                   min_cell_area_um2=300, junction_search_radius_um=1.0, min_gap_area_um2=4):
    """
    Measure every cell and every gap in one image.

    Parameters
    ----------
    stack_path : pathlib.Path
        Z-stack tif, (Z, C, Y, X), with the pixel size in its metadata.
    labels_dir : pathlib.Path
        Folder with the cell labels: one tif per image, named "combined_vecad_" + the stack's file name.
    output_dir : pathlib.Path
        QC figures are saved to output_dir/qc.
    to_plot : bool, optional (default=True)
        Whether to save a QC figure.
    dapi_channel, vecad_channel, icam1_channel : int, optional (default=0, 1, 2)
        Channel indices in the stack.
    min_cell_area_um2 : float, optional (default=300)
        Labels smaller than this are fragments and are merged into a neighbouring cell (see merge_small_fragments).
    junction_search_radius_um : float, optional (default=1.0)
        How far from each cell outline to look for the VE-Cad junction, in µm.
    min_gap_area_um2 : float, optional (default=4)
        Smallest gap that is counted, in µm².

    Returns
    -------
    per_cell : pandas.DataFrame
        One row per cell that does not touch the image border:
        - 'filename', 'experiment', 'treatment', 'cell_label'
        - 'cell_area_um2'
        - 'cell_aspect_ratio': longest / shortest axis of the ellipse fitted to the cell (1 = round)
        - 'ICAM1_total', 'VECad_total': summed over every z slice inside the cell
        - 'VECad_junction': VE-Cad junction brightness (see measure_junction_vecad)
    per_image : pandas.DataFrame
        One row:
        - 'filename', 'experiment', 'treatment'
        - 'n_cells': all cells in the image, including those touching the border
        - 'gap_area_pct': % of the monolayer (cells + gaps) that is gap
        - 'total_gap_area_um2'
    """
    filename = stack_path.stem
    with tifffile.TiffFile(stack_path) as tif:
        stack = tif.series[0].asarray()  # (Z, C, Y, X)
        z_um, y_um, x_um = tif_pixel_sizes(tif)
    pixel_area_um2 = x_um * y_um

    # Clean up the labels first: merge fragments into their neighbouring cell, then fill holes inside cells
    cell_labels = tifffile.imread(labels_dir / f"combined_vecad_{stack_path.name}").astype(np.int32)
    cell_labels = merge_small_fragments(cell_labels, pixel_area_um2, min_cell_area_um2)
    cell_labels = fill_holes_in_cells(cell_labels)

    # Sum over z: the total signal through the whole cell, not just its brightest slice
    dapi = stack[:, dapi_channel].sum(axis=0, dtype=np.float64)
    vecad = stack[:, vecad_channel].sum(axis=0, dtype=np.float64)
    icam1 = stack[:, icam1_channel].sum(axis=0, dtype=np.float64)

    ################## Cells ##################################################################################
    measured_labels = clear_border(cell_labels)  # cells cut off by the image border are only partly visible
    per_cell = pd.DataFrame(regionprops_table(measured_labels, properties=("label", "area", "axis_major_length", "axis_minor_length")))
    per_cell = per_cell.rename(columns={"label": "cell_label"})
    cell_ids = per_cell["cell_label"].to_numpy()
    per_cell["cell_area_um2"] = per_cell["area"] * pixel_area_um2
    per_cell["cell_aspect_ratio"] = per_cell["axis_major_length"] / per_cell["axis_minor_length"]
    per_cell["ICAM1_total"] = ndi.sum_labels(icam1, measured_labels, index=cell_ids)
    per_cell["VECad_total"] = ndi.sum_labels(vecad, measured_labels, index=cell_ids)
    per_cell["VECad_junction"] = measure_junction_vecad(vecad, measured_labels, cell_ids,
                                                        search_radius_px=round(junction_search_radius_um / x_um))
    per_cell = add_image_details(per_cell, filename)
    per_cell = per_cell[["filename", "experiment", "treatment", "cell_label",
                         "cell_area_um2", "cell_aspect_ratio", "ICAM1_total", "VECad_total", "VECad_junction"]]

    ################## Gaps ###################################################################################
    gap_labels, monolayer = find_gaps(cell_labels, pixel_area_um2, min_gap_area_um2)
    gap_area_pct = 100 * (gap_labels > 0).sum() / monolayer.sum()
    per_image = add_image_details(pd.DataFrame([{
        "n_cells": len(np.unique(cell_labels[cell_labels > 0])),
        "gap_area_pct": gap_area_pct,
        "total_gap_area_um2": (gap_labels > 0).sum() * pixel_area_um2,
    }]), filename)
    per_image = per_image[["filename", "experiment", "treatment", "n_cells", "gap_area_pct", "total_gap_area_um2"]]

    if to_plot:
        plot_qc(output_dir, filename, dapi, vecad, icam1, cell_labels, measured_labels, gap_labels, monolayer, per_cell, gap_area_pct)

    return per_cell, per_image

# Run

In [ ]:
root = pathlib.Path(r"C:\Users\taylorhearn\Dropbox\Dorota_Image_Analzsis_Size_Spaces")
stack_dir = root / "Input_Reordered_C1_Dapi_C2_VeCad_C3_ICAM1_C4_Other"  # C1 DAPI, C2 VE-Cad, C3 ICAM1
labels_dir = root / "C1_Dapi_C2_VecCad_C3_ICAM1" / "NEWATTEMPT" / "All_Labels_dapi_vecad_icam1_perchannel"
output_dir = root / "Gap_Analysis_Outputs"
(output_dir / "qc").mkdir(parents=True, exist_ok=True)

stack_paths = sorted(stack_dir.glob("*.tif"))
print(len(stack_paths), "images")

In [ ]:
# Test on one image
per_cell, per_image = quantify_image(stack_paths[0], labels_dir, output_dir)
fig, ax = plt.subplots(figsize=(18, 12))
ax.imshow(plt.imread(output_dir / "qc" / f"{stack_paths[0].stem}_qc.png"))
ax.axis("off")
plt.show()
per_image

In [ ]:
with tqdm_joblib(tqdm(desc="Image Analysis", total=len(stack_paths))) as progress_bar:
    output = Parallel(n_jobs=4)(delayed(quantify_image)(stack_path, labels_dir, output_dir) for stack_path in stack_paths)

per_cell = pd.concat([cells for cells, image in output], ignore_index=True)
per_image = pd.concat([image for cells, image in output], ignore_index=True)
per_cell.to_csv(output_dir / "monolayer_gap_analysis_per_cell.csv", index=False)
per_image.to_csv(output_dir / "monolayer_gap_analysis_per_image.csv", index=False)
print("saved to", output_dir)
per_image